In [0]:
from pyspark.sql.functions import *

### **Parameters**

In [0]:
#catalog
catalog="workspace"

#source schema
source_schema="silver"

#source object
source_object="silver_bookings"
#CDC column
cdc_col="modifiedDate"

#Backdated referesh
backdated_refresh=""

#Source fact table
fact_table=f"{catalog}.{source_schema}.{source_object}"
#source schema
source_schema="silver"
#target object
target_object="factBookings"
#target schema
target_schema="gold"
#Fact key col list
fact_key_cols=["DimPassengersKey","DimFlightsKey","DimAirportsKey",'booking_date']

In [0]:
dimensions=[
    {
        "table":f"{catalog}.{target_schema}.dimPassengers",
        "alias":"dimPassengers",
        "join_keys":[("passenger_id","passenger_id")] #(fact_col,dim_col)
    },
    {
        "table":f"{catalog}.{target_schema}.dimFlights",
        "alias":"dimFlights",
        "join_keys":[("flight_id","flight_id")] #(fact_col,dim_col)
    },
    {
        "table":f"{catalog}.{target_schema}.dimAirports",
        "alias":"dimAirports",
        "join_keys":[("airport_id","airport_id")] #(fact_col,dim_col)
    }
]

#columns you want to keep from fact table besides the surrogate keys
fact_columns=["amount","booking_date","modifiedDate"]

####**Last Load Date** 

In [0]:
#NO Backdated Refresh
if len(backdated_refresh) == 0:
  #If table existin the destination
  if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    last_load= spark.sql(f"SELECT max({cdc_col}) FROM workspace.{target_schema}.{target_object}").collect()[0][0]
  else:
    last_load = "1990-01-01 00:00:00"
#Yes backdated refresh    
else:
  last_load=backdated_refresh
#Test the last load
last_load

####**Dynamic Fact Query** 

In [0]:
def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_col, last_load):
    fact_alias = "f"

    # Base columns to select
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    # Build joins dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]
        table_name = table_full.split(".")[-1]
        surrogate_key = f"{alias}.{table_name}Key"
        select_cols.append(surrogate_key)

        # Build ON clause
        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}"
            for fk, dk in dim["join_keys"]
        ]

        join_clause = f"LEFT JOIN {table_full} {alias} ON {' AND '.join(on_conditions)}"
        join_clauses.append(join_clause)

    # Final select and JOIN clauses
    select_clause = ",\n    ".join(select_cols)
    join = "\n".join(join_clauses)

    # WHERE clause using last_load
    where_clause = f"{fact_alias}.{cdc_col} >= '{last_load}'"

    # Final query
    query = f"""
SELECT
    {select_clause}
FROM
    {fact_table} {fact_alias}
{join}
WHERE
    {where_clause}
""".strip()

    return query

In [0]:
#Generate query
query=generate_fact_query_incremental(fact_table,dimensions,fact_columns,cdc_col,last_load)

print(query)

#### **DF_FACT**

In [0]:
df_fact=spark.sql(query)

In [0]:
df_fact.display()

### **UPSERT**

In [0]:
#Fact Key Columns Merge Condition
fact_key_cols

In [0]:
fact_key_cols_str=" AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])
fact_key_cols_str


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    
    dlt_obj.alias('trg').merge(
        df_fact.alias('src'),
        fact_key_cols_str
    ) \
    .whenMatchedUpdateAll(
        condition=f"src.{cdc_col} >= trg.{cdc_col}"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

else:
    df_fact.write.format('delta') \
        .mode('append') \
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")
                           

In [0]:
%sql
SELECT * FROM workspace.gold.factbookings

In [0]:
#Checking duplicates in dim tables
#df=spark.sql("SELECT* FROM workspace.gold.dimAirports").groupby("DimAirportsKey").count().filter(col('count')>1)
#df.display()
